# v2.0 Fast Demo：建图（gap=3）+ `QueryConfig.fast()`

1. 从 `test_data` 截取 **2 min** 片段  
2. `rig_pairs`：每秒 1 keyframe，**`max_temporal_gap=3`**（无 cross-camera）  
3. `feature_pairs`：DISK + LightGlue + INS 三角化 → `sfm/`、`global_features.h5`  
4. 均匀选 **20** 张未建图帧 → Fast query（NetVLAD top‑k + **NN-mutual** → PnP）  

**Kernel：** `SchubKI v2.0` · `source v2.0/scripts/env.sh`  
**输出：** `test/outputs/segment_2min_gap3_rig/`


## 0. 环境与路径

In [1]:
import os
import sys
import time
from pathlib import Path

import numpy as np

TEST_DIR = Path.cwd().resolve()
V2_ROOT = TEST_DIR.parent
sys.path.insert(0, str(V2_ROOT / "src"))

HLOC_ROOT = V2_ROOT / "third_party" / "Hierarchical-Localization"
sys.path.insert(0, str(HLOC_ROOT))
os.environ.setdefault("HLOC_ROOT", str(HLOC_ROOT))
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

torch_hub = V2_ROOT / "model_cache" / "torch_hub"
torch_hub.mkdir(parents=True, exist_ok=True)
import torch

torch.hub.set_dir(str(torch_hub))

from robotcar.paths import DEFAULT_LEFT_IMAGES, DEFAULT_RIGHT_IMAGES
from robotcar.stereo import sync_stereo_pairs

OUTPUT_DIR = V2_ROOT / "test" / "outputs" / "segment_2min_gap3_rig"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUTPUT:", OUTPUT_DIR)

OUTPUT: /Users/hbai/SchubKI/v2.0/test/outputs/segment_2min_gap3_rig


## 1. 截取 2 分钟 ts 序列

In [2]:
SEGMENT_DURATION_US = 120_000_000  # 2 min
KEYFRAME_INTERVAL_US = 1_000_000  # 1 Hz
MAX_TEMPORAL_GAP = 3

synced = sync_stereo_pairs(DEFAULT_LEFT_IMAGES, DEFAULT_RIGHT_IMAGES)
all_ts = [p[0] for p in synced]
print(f"全序列: {len(all_ts)} 帧, 时长 {(all_ts[-1] - all_ts[0]) / 1e6:.1f} s")

START_TS = all_ts[0]
END_TS = START_TS + SEGMENT_DURATION_US
segment_ts = [t for t in all_ts if START_TS <= t < END_TS]
print(
    f"片段: start={START_TS}, 帧数={len(segment_ts)}, "
    f"时长≈{(segment_ts[-1] - segment_ts[0]) / 1e6:.1f} s"
)
print("max_temporal_gap:", MAX_TEMPORAL_GAP)

全序列: 2929 帧, 时长 192.0 s
片段: start=1400075224469383, 帧数=1794, 时长≈120.0 s
max_temporal_gap: 3


## 2. `rig_pairs`（`max_temporal_gap=3`，无 cross-camera）

In [3]:
from rig_pairs import KeyframeConfig, PairsConfig, generate_pairs_for_test_data

kf_cfg = KeyframeConfig(
    start_ts=START_TS,
    segment_duration_us=SEGMENT_DURATION_US,
    keyframe_interval_us=KEYFRAME_INTERVAL_US,
)
pc_cfg = PairsConfig(
    max_temporal_gap=MAX_TEMPORAL_GAP,
    min_baseline_m=0.75,
    include_cross_camera=False,
)
keyframes, pair_lists, pair_paths = generate_pairs_for_test_data(
    output_dir=OUTPUT_DIR,
    keyframe_config=kf_cfg,
    pairs_config=pc_cfg,
)
print(f"Keyframes: {len(keyframes)}, pairs-all: {len(pair_lists.all_pairs)}")
print(
    f"  stereo={len(pair_lists.stereo)}, "
    f"temporal_L={len(pair_lists.temporal_left)}, "
    f"temporal_R={len(pair_lists.temporal_right)}, "
    f"cross={len(pair_lists.cross)}"
)
for k in keyframes[:3]:
    print(f"  [{k.index}] ts={k.ts}")
print("  ...")
print(f"  [{keyframes[-1].index}] ts={keyframes[-1].ts}")

Keyframes: 113, pairs-all: 761
  stereo=113, temporal_L=324, temporal_R=324, cross=0
  [0] ts=1400075224469383
  [1] ts=1400075225531740
  [2] ts=1400075226594096
  ...
  [112] ts=1400075343890599


## 3. `feature_pairs` → SfM

DISK + LightGlue（建图仍用 LightGlue）+ INS 三角化 + NetVLAD。  
续跑可设 `overwrite_features=False`。

In [4]:
from hloc.utils.device import inference_device
from feature_pairs import FeaturePairsConfig, FeaturePairsPaths, run_feature_pairs_pipeline

print("DISK / LightGlue device:", inference_device())

t0 = time.perf_counter()
fp_cfg = FeaturePairsConfig(
    paths=FeaturePairsPaths(rig_sfm_dir=OUTPUT_DIR),
    overwrite_features=False,
    build_reference=True,
    run_triangulation=True,
    build_global_descriptors=True,
)
fp_result = run_feature_pairs_pipeline(fp_cfg)
print(f"feature_pairs elapsed: {time.perf_counter() - t0:.1f} s")
print(fp_result.graph)
if fp_result.validation:
    print("points3D:", fp_result.validation.num_points3D)
    print("mean reproj:", fp_result.validation.mean_reprojection_error)
model = fp_result.model
assert model is not None

[2026/10/15 20:16:01 hloc INFO] Extracting features (cached model):
{'name': 'disk', 'max_keypoints': 5000}
[2026/10/15 20:16:01 hloc INFO] Skipping the extraction.
[2026/10/15 20:16:01 hloc INFO] Matching features (cached model):
{'name': 'lightglue', 'features': 'disk'}
[2026/10/15 20:16:01 hloc INFO] Skipping the matching.
[2026/10/15 20:16:01 hloc INFO] Extracting features (cached model):
{'name': 'netvlad'}
[2026/10/15 20:16:02 hloc INFO] Skipping the extraction.


DISK / LightGlue device: mps


[2026/10/15 20:16:02 hloc WARNING] The database already exists, deleting it.
[2026/10/15 20:16:02 hloc INFO] Importing features into the database...
100%|██████████| 226/226 [00:00<00:00, 2552.03it/s]
[2026/10/15 20:16:02 hloc INFO] Importing matches into the database...
100%|██████████| 761/761 [00:00<00:00, 3281.43it/s]
[2026/10/15 20:16:02 hloc INFO] Performing geometric verification of the matches...
100%|██████████| 223/223 [00:02<00:00, 99.88it/s] 
[2026/10/15 20:16:04 hloc INFO] mean/med/min/max valid matches 91.57/99.37/0.00/100.00%.
[2026/10/15 20:16:04 hloc INFO] Running 3D triangulation...
[2026/10/15 20:16:08 hloc INFO] Finished the triangulation with statistics:
Reconstruction:
	num_rigs = 2
	num_cameras = 2
	num_frames = 226
	num_reg_frames = 226
	num_images = 226
	num_points3D = 26501
	num_observations = 76462
	mean_track_length = 2.88525
	mean_observations_per_image = 338.327
	mean_reprojection_error = 1.74887


feature_pairs elapsed: 7.7 s
KeyframeGraphSummary(num_keyframes=113, num_images=226, num_pairs=761, num_stereo_pairs=113, num_temporal_pairs=648, image_names=['left/1400075224469383.png', 'left/1400075225531740.png', 'left/1400075226594096.png', 'left/1400075227656425.png', 'left/1400075228718755.png', 'left/1400075229781087.png', 'left/1400075230843416.png', 'left/1400075231905828.png', 'left/1400075232968183.png', 'left/1400075234030574.png', 'left/1400075235092934.png', 'left/1400075236155266.png', 'left/1400075237217596.png', 'left/1400075238279926.png', 'left/1400075239342257.png', 'left/1400075240404585.png', 'left/1400075241466938.png', 'left/1400075242529297.png', 'left/1400075243591648.png', 'left/1400075244654006.png', 'left/1400075245778857.png', 'left/1400075246903707.png', 'left/1400075247966056.png', 'left/1400075249028432.png', 'left/1400075250090810.png', 'left/1400075251153141.png', 'left/1400075252278015.png', 'left/1400075253340430.png', 'left/1400075254402836.png', 

## 3b. 稀疏地图（可选）

In [5]:
from hloc.utils import viz_3d

fig_map = viz_3d.init_figure()
viz_3d.plot_reconstruction(
    fig_map, model, color="rgba(255,0,0,0.45)", name="mapping", points_rgb=True
)
fig_map.show()

## 4. 选取 20 张未参与建图的 query 图

从片段中排除 keyframe 时间戳后，**均匀采样 20 帧**（左目）。


In [6]:
NUM_QUERIES = 20
QUERY_CAMERA = "left"

keyframe_ts = {kf.ts for kf in keyframes}
candidates = [t for t in segment_ts if t not in keyframe_ts]
if not candidates:
    raise RuntimeError("No non-keyframe timestamps in segment; loosen keyframe interval")

if len(candidates) <= NUM_QUERIES:
    QUERY_TS_LIST = list(candidates)
else:
    idx = np.linspace(0, len(candidates) - 1, NUM_QUERIES, dtype=int)
    QUERY_TS_LIST = [candidates[i] for i in idx]

print(f"candidates (non-keyframe): {len(candidates)}")
print(f"selected queries: {len(QUERY_TS_LIST)}  camera={QUERY_CAMERA}")
print("first/last:", QUERY_TS_LIST[0], QUERY_TS_LIST[-1])


candidates (non-keyframe): 1681
selected queries: 20  camera=left
first/last: 1400075224531875 1400075344453058


## 5. Fast query（对 20 帧跑 `QueryConfig.fast()`）

- `retrieval_top_k=3`，`NN-mutual`
- 汇总成功率 / 位姿误差 / 耗时；`q_result` 取中间一帧供 §7 可视化


In [7]:
from query import QueryConfig, QueryImageSpec, QueryWorkspacePaths, run_query_localization

paths = QueryWorkspacePaths(rig_sfm_dir=OUTPUT_DIR)
q_cfg = QueryConfig.fast(paths, retrieval_top_k=3)

q_results = []
rows = []
t_all = time.perf_counter()
for i, ts in enumerate(QUERY_TS_LIST):
    t0 = time.perf_counter()
    r = run_query_localization(
        QueryImageSpec(timestamp_us=ts, camera=QUERY_CAMERA),
        q_cfg,
    )
    dt = time.perf_counter() - t0
    q_results.append(r)
    err = None
    if r.predicted.success and r.predicted.cam_from_world is not None:
        err = float(
            np.linalg.norm(
                np.asarray(r.predicted.cam_from_world.translation)
                - np.asarray(r.gt_cam_from_world.translation)
            )
        )
    rows.append(
        {
            "i": i,
            "ts": ts,
            "ok": r.predicted.success,
            "inliers": r.predicted.num_inliers if r.predicted.success else 0,
            "corr": r.predicted.num_correspondences if r.predicted.success else 0,
            "err_m": err,
            "dt_s": dt,
        }
    )
    err_s = f"{err:.3f} m" if err is not None else "-"
    print(
        f"[{i+1:02d}/{len(QUERY_TS_LIST)}] ts={ts}  {dt:.1f}s  "
        f"ok={r.predicted.success}  inliers={rows[-1]['inliers']}/{rows[-1]['corr']}  err={err_s}"
    )

total = time.perf_counter() - t_all
n_ok = sum(1 for x in rows if x["ok"])
errs = [x["err_m"] for x in rows if x["err_m"] is not None]
print(f"\nTotal: {total:.1f} s  mean/query: {total/len(QUERY_TS_LIST):.1f} s")
print(f"Success: {n_ok}/{len(QUERY_TS_LIST)}")
if errs:
    print(f"Position error [m]: mean={np.mean(errs):.3f}  median={np.median(errs):.3f}  max={np.max(errs):.3f}")

# mid query for 3D viz
q_result = q_results[len(q_results) // 2]
print(f"\nViz query (mid): ts={QUERY_TS_LIST[len(QUERY_TS_LIST)//2]}  inliers={q_result.predicted.num_inliers}")


[2026/10/15 20:16:32 hloc INFO] Extracting features (cached model):
{'name': 'netvlad'}
[2026/10/15 20:16:32 hloc INFO] Skipping the extraction.
[2026/10/15 20:16:32 hloc INFO] Extracting image pairs from a retrieval database.
[2026/10/15 20:16:33 hloc INFO] Found 3 pairs.
[2026/10/15 20:16:33 hloc INFO] Extracting features (cached model):
{'name': 'disk', 'max_keypoints': 5000}
[2026/10/15 20:16:33 hloc INFO] Skipping the extraction.
[2026/10/15 20:16:33 hloc INFO] Matching features (cached model):
{'name': 'nearest_neighbor', 'do_mutual_check': True}
[2026/10/15 20:16:33 hloc INFO] Skipping the matching.
[2026/10/15 20:16:33 hloc INFO] Extracting features (cached model):
{'name': 'netvlad'}
[2026/10/15 20:16:33 hloc INFO] Skipping the extraction.
[2026/10/15 20:16:33 hloc INFO] Extracting image pairs from a retrieval database.
[2026/10/15 20:16:33 hloc INFO] Found 3 pairs.
[2026/10/15 20:16:33 hloc INFO] Extracting features (cached model):
{'name': 'disk', 'max_keypoints': 5000}
[202

[01/20] ts=1400075224531875  0.6s  ok=False  inliers=0/0  err=-
[02/20] ts=1400075230343495  0.2s  ok=True  inliers=722/739  err=0.041 m


[2026/10/15 20:16:33 hloc INFO] Extracting features (cached model):
{'name': 'netvlad'}
[2026/10/15 20:16:33 hloc INFO] Skipping the extraction.
[2026/10/15 20:16:33 hloc INFO] Extracting image pairs from a retrieval database.
[2026/10/15 20:16:33 hloc INFO] Found 3 pairs.
[2026/10/15 20:16:33 hloc INFO] Extracting features (cached model):
{'name': 'disk', 'max_keypoints': 5000}
[2026/10/15 20:16:33 hloc INFO] Skipping the extraction.
[2026/10/15 20:16:33 hloc INFO] Matching features (cached model):
{'name': 'nearest_neighbor', 'do_mutual_check': True}
[2026/10/15 20:16:33 hloc INFO] Skipping the matching.
[2026/10/15 20:16:34 hloc INFO] Extracting features (cached model):
{'name': 'netvlad'}
[2026/10/15 20:16:34 hloc INFO] Skipping the extraction.
[2026/10/15 20:16:34 hloc INFO] Extracting image pairs from a retrieval database.
[2026/10/15 20:16:34 hloc INFO] Found 3 pairs.
[2026/10/15 20:16:34 hloc INFO] Extracting features (cached model):
{'name': 'disk', 'max_keypoints': 5000}
[202

[03/20] ts=1400075236217757  0.3s  ok=True  inliers=576/880  err=0.070 m


[2026/10/15 20:16:34 hloc INFO] Extracting features (cached model):
{'name': 'netvlad'}
[2026/10/15 20:16:34 hloc INFO] Skipping the extraction.
[2026/10/15 20:16:34 hloc INFO] Extracting image pairs from a retrieval database.
[2026/10/15 20:16:34 hloc INFO] Found 3 pairs.
[2026/10/15 20:16:34 hloc INFO] Extracting features (cached model):
{'name': 'disk', 'max_keypoints': 5000}
[2026/10/15 20:16:34 hloc INFO] Skipping the extraction.
[2026/10/15 20:16:34 hloc INFO] Matching features (cached model):
{'name': 'nearest_neighbor', 'do_mutual_check': True}
[2026/10/15 20:16:34 hloc INFO] Skipping the matching.


...

[18/20] ts=1400075330517400  0.3s  ok=True  inliers=169/221  err=0.753 m


[2026/10/15 20:16:37 hloc INFO] Skipping the extraction.
[2026/10/15 20:16:37 hloc INFO] Matching features (cached model):
{'name': 'nearest_neighbor', 'do_mutual_check': True}
[2026/10/15 20:16:37 hloc INFO] Skipping the matching.
[2026/10/15 20:16:38 hloc INFO] Extracting features (cached model):
{'name': 'netvlad'}
[2026/10/15 20:16:38 hloc INFO] Skipping the extraction.
[2026/10/15 20:16:38 hloc INFO] Extracting image pairs from a retrieval database.
[2026/10/15 20:16:38 hloc INFO] Found 3 pairs.


[19/20] ts=1400075337328981  0.3s  ok=True  inliers=141/272  err=0.228 m


[2026/10/15 20:16:38 hloc INFO] Extracting features (cached model):
{'name': 'disk', 'max_keypoints': 5000}
[2026/10/15 20:16:38 hloc INFO] Skipping the extraction.
[2026/10/15 20:16:38 hloc INFO] Matching features (cached model):
{'name': 'nearest_neighbor', 'do_mutual_check': True}
[2026/10/15 20:16:38 hloc INFO] Skipping the matching.


[20/20] ts=1400075344453058  0.3s  ok=True  inliers=44/179  err=8.845 m

Total: 5.3 s  mean/query: 0.3 s
Success: 19/20
Position error [m]: mean=2.417  median=1.097  max=8.845

Viz query (mid): ts=1400075285336024  inliers=433


## 6. 全部 query：位姿表 + 3D（地图点 + GT/预测相机锥）

- 打印每帧 GT / predicted 的 translation、quaternion、位置误差  
- Plotly：SfM 三维点 + 全部 query 的 GT（蓝）与预测（绿）相机锥（`size=4`）


In [8]:
from hloc.utils import viz_3d

CAMERA_FRUSTUM_SIZE = 4.0  # larger cones (hloc default ~1)


def _pose_parts(pose):
    t = np.asarray(pose.translation, dtype=float)
    q = np.asarray(pose.rotation.quat, dtype=float)
    return t, q


def _print_pose(label: str, pose) -> None:
    t, q = _pose_parts(pose)
    print(f"{label}")
    print(f"  translation [m]:  [{t[0]:+.4f}, {t[1]:+.4f}, {t[2]:+.4f}]")
    print(f"  quaternion wxyz:  [{q[0]:+.6f}, {q[1]:+.6f}, {q[2]:+.6f}, {q[3]:+.6f}]")


def _enable_legend_last_trace(fig, name: str) -> None:
    """hloc plot_camera sets showlegend=False; re-enable one entry per group."""
    if fig.data:
        fig.data[-1].showlegend = True
        fig.data[-1].name = name


print(f"{'i':>3}  {'ts':>16}  {'ok':>3}  {'err_m':>8}  {'inliers':>12}")
print("-" * 72)
for i, (ts, r) in enumerate(zip(QUERY_TS_LIST, q_results)):
    print(f"\n=== query [{i+1:02d}/{len(q_results)}] ts={ts} camera={QUERY_CAMERA} ===")
    _print_pose("GT", r.gt_cam_from_world)
    if r.predicted.success and r.predicted.cam_from_world is not None:
        _print_pose("Predicted", r.predicted.cam_from_world)
        gt_t, _ = _pose_parts(r.gt_cam_from_world)
        pr_t, _ = _pose_parts(r.predicted.cam_from_world)
        err = float(np.linalg.norm(pr_t - gt_t))
        print(f"  position error: {err:.4f} m")
        print(
            f"  PnP inliers: {r.predicted.num_inliers}/"
            f"{r.predicted.num_correspondences}"
        )
        ok, err_s, inl = (
            "Y",
            f"{err:.4f}",
            f"{r.predicted.num_inliers}/{r.predicted.num_correspondences}",
        )
    else:
        print("Predicted: FAILED")
        ok, err_s, inl = "N", "-", "0/0"
    print(f"[{i+1:02d}] ts={ts}  ok={ok}  err_m={err_s}  inliers={inl}")

errs = []
for r in q_results:
    if r.predicted.success and r.predicted.cam_from_world is not None:
        errs.append(
            float(
                np.linalg.norm(
                    np.asarray(r.predicted.cam_from_world.translation)
                    - np.asarray(r.gt_cam_from_world.translation)
                )
            )
        )
n_ok = sum(1 for r in q_results if r.predicted.success)
print("\n=== aggregate ===")
print(f"Success: {n_ok}/{len(q_results)}")
if errs:
    print(
        f"Position error [m]: mean={np.mean(errs):.4f}  "
        f"median={np.median(errs):.4f}  max={np.max(errs):.4f}"
    )

fig_q = viz_3d.init_figure()
viz_3d.plot_reconstruction(
    fig_q,
    model,
    color="rgba(180,180,180,0.35)",
    name="SfM points",
    points_rgb=True,
    cameras=False,
    points=True,
)

gt_legend_done = False
pred_legend_done = False
for i, r in enumerate(q_results):
    viz_3d.plot_camera_colmap(
        fig_q,
        r.gt_cam_from_world,
        r.gt_camera,
        color="rgba(0,100,255,0.55)",
        name="GT (queries)",
        legendgroup="GT (queries)",
        fill=True,
        size=CAMERA_FRUSTUM_SIZE,
        text=f"GT {QUERY_TS_LIST[i]}",
    )
    if not gt_legend_done:
        _enable_legend_last_trace(fig_q, "GT (queries)")
        gt_legend_done = True

    if r.predicted.success and r.predicted.cam_from_world is not None:
        pred_cam = r.gt_camera
        if r.pnp_ret and r.pnp_ret.get("camera") is not None:
            pred_cam = r.pnp_ret["camera"]
        viz_3d.plot_camera_colmap(
            fig_q,
            r.predicted.cam_from_world,
            pred_cam,
            color="rgba(0,220,0,0.55)",
            name="Predicted (queries)",
            legendgroup="Predicted (queries)",
            fill=True,
            size=CAMERA_FRUSTUM_SIZE,
            text=f"Pred {QUERY_TS_LIST[i]}",
        )
        if not pred_legend_done:
            _enable_legend_last_trace(fig_q, "Predicted (queries)")
            pred_legend_done = True

fig_q.show()


  i                ts   ok     err_m       inliers
------------------------------------------------------------------------

=== query [01/20] ts=1400075224531875 camera=left ===
GT
  translation [m]:  [+0.1208, +1.1951, -1.7165]
  quaternion wxyz:  [-0.494361, -0.485436, -0.511727, +0.508030]
Predicted: FAILED
[01] ts=1400075224531875  ok=N  err_m=-  inliers=0/0

=== query [02/20] ts=1400075230343495 camera=left ===
GT
  translation [m]:  [+0.3452, +1.1430, -4.1048]
  quaternion wxyz:  [+0.484615, +0.499475, +0.527882, -0.486841]
Predicted
  translation [m]:  [+0.3784, +1.1599, -4.1210]
  quaternion wxyz:  [+0.484184, +0.499627, +0.528241, -0.486723]
  position error: 0.0406 m
  PnP inliers: 722/739
[02] ts=1400075230343495  ok=Y  err_m=0.0406  inliers=722/739

=== query [03/20] ts=1400075236217757 camera=left ===
GT
  translation [m]:  [+1.3098, +2.0022, -26.0597]
  quaternion wxyz:  [+0.483232, +0.495297, +0.556792, -0.459511]
Predicted
  translation [m]:  [+1.3191, +2.0708, -26.048